# 01 - Race pace and consistency (single race)

**Questions**
1. Who had the fastest race pace once traffic, pit laps and neutralised laps are removed?
2. Which drivers were the most consistent?
3. How does tyre compound show up in the lap times?

**Before running:** load the race with
`python -m src.ingest_jolpica <season> <round>` then `python -m src.ingest_fastf1 <season> <round>`.

**Definitions**
- *Clean lap*: not lap 1, not an in/out lap, green flag, accurate timing (the `is_clean` flag).
- *Outlier*: a clean lap more than 2% slower than its stint trend line (traffic, mistakes). Removed.
- *Race pace*: median clean lap time.
- *Consistency*: lap-time spread **after removing the linear trend inside each stint**. Raw spread
  would penalise a driver for fuel burn and tyre wear, which are not inconsistency. Lower is better.

In [ ]:
import sys
from pathlib import Path

# make `src` importable whether the notebook is run from the repo root or notebooks/
root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import pandas as pd

from src.analysis import load_race_laps, clean_laps, stint_stats, driver_summary
from src.db import get_connection
from src.plots import (plot_pace_distribution, plot_consistency, plot_stints,
                       plot_pace_vs_consistency, summary_table)

pd.set_option("display.width", 140)

SEASON, ROUND = 2024, 1   # change these to look at another race

## 1. Load laps and apply the cleaning rules

In [ ]:
conn = get_connection()
laps = load_race_laps(conn, SEASON, ROUND)
clean = clean_laps(laps)

print(f"Total laps recorded : {len(laps):>5}")
print(f"Flagged clean       : {int(laps['is_clean'].sum()):>5}")
print(f"After outlier filter: {len(clean):>5}")
print(f"Drivers             : {laps['code'].nunique():>5}")

**Sanity check:** a typical race keeps roughly 60-80% of laps. If far fewer survive, check the track
status codes and the `is_accurate` flag before trusting any result below.

In [ ]:
stints = stint_stats(clean)
summary = driver_summary(clean, stints)
summary_table(summary)

## 2. Race pace

In [ ]:
fig = plot_pace_distribution(clean, summary)
plt.show()

Each dot is one clean lap; the dark tick is the driver's median. Drivers are ordered fastest to slowest.
A wide cloud means the driver's laps varied a lot, which the next section quantifies.

## 3. Consistency

In [ ]:
fig = plot_consistency(summary)
plt.show()

In [ ]:
fig = plot_pace_vs_consistency(summary)
plt.show()

Bottom-left is fast *and* consistent. A driver who is quick but spread out may have been
managing traffic or tyres; a slow, tight driver may have been circulating to a target.

## 4. Stints and tyre compound

In [ ]:
top = summary["code"].head(4).tolist()   # four fastest drivers by median pace
fig = plot_stints(clean, top)
plt.show()

In [ ]:
# Degradation by compound: seconds gained/lost per lap within a stint
# (tyre wear and fuel burn pull in opposite directions, so this is the net trend)
(stints.groupby("compound")
       .agg(stints=("stint", "size"),
            avg_slope_s_per_lap=("slope_s_per_lap", "mean"),
            median_pace_s=("median_s", "median"))
       .round(3))

## 5. Findings (fill in after running a real race)

- **Fastest race pace:**
- **Most consistent:**
- **Largest gap between pace rank and finishing position, and why:**
- **Compound that degraded most:**
- **Caveats:** one race is a small sample; fuel load is not corrected explicitly, only trended out.